# Streaming Weather Data Producer

This notebook implements a Kafka producer that simulates a live weather-sensor feed for the building energy consumption prediction pipeline. It reads historical weather records, converts each row to JSON, and publishes batches of messages to Kafka for downstream Spark Structured Streaming processing.

## Overview

The producer forms the ingestion layer of the real-time analytics pipeline. It prepares weather observations as event messages so that the streaming model can consume them continuously and generate energy-consumption predictions.

## Pipeline Context

**Weather CSV → Kafka Producer → Kafka Topic → Spark Structured Streaming → ML Prediction → Consumer Visualisation**

The producer intentionally avoids Spark processing so it behaves like a lightweight sensor publisher. Each Kafka message contains weather features and an event timestamp used later for event-time processing and watermarking.

## Package Installation

The producer uses `kafka-python` to connect to Kafka, create topics, serialise records, and publish messages. Install this package before running the notebook in a new environment.

In [ ]:
# Install kafka-python
!pip install kafka-python --break-system-packages

## Imported Libraries

The notebook uses Kafka utilities for publishing messages, pandas for reading the weather CSV, JSON serialisation for Kafka payloads, and time utilities to simulate streaming intervals.

In [ ]:
from kafka import KafkaProducer, KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError
import pandas as pd
import json
import time
from datetime import datetime

print(" Libraries imported successfully!")

## Configuration

This section defines the Kafka broker, target topic, source weather file path, and batch controls. Keep these settings centralised so the notebook can be adapted easily for local Docker, local machine, or cloud environments.

In [ ]:
# Kafka broker address (for Docker container connecting to host)
KAFKA_BROKER = 'host.docker.internal:9092'

# Kafka topic name
TOPIC_NAME = 'weather_stream'

# Path to weather.csv file (update this path if needed!)
WEATHER_CSV_PATH = 'weather.csv'

# Project requirements
DAYS_PER_BATCH = 5              # 5 days per batch
RECORDS_PER_DAY = 24            # 24 hourly records per day
RECORDS_PER_BATCH = 120         # Total: 5 * 24 = 120
BATCH_INTERVAL = 5              # Wait 5 seconds between batches

print("Configuration:")
print(f"  Kafka Broker: {KAFKA_BROKER}")
print(f"  Topic: {TOPIC_NAME}")
print(f"  Records per batch: {RECORDS_PER_BATCH}")
print(f"  Batch interval: {BATCH_INTERVAL} seconds")

## Kafka Topic Creation

A dedicated Kafka topic named `weather_stream` is created for the weather feed. The downstream Spark Structured Streaming notebook subscribes to this topic and parses each message as a weather event.

In [ ]:
try:
    # Create admin client
    admin_client = KafkaAdminClient(
        bootstrap_servers=KAFKA_BROKER,
        client_id='weather_producer_admin'
    )
    
    # Create topic
    topic = NewTopic(name=TOPIC_NAME, num_partitions=1, replication_factor=1)
    admin_client.create_topics(new_topics=[topic], validate_only=False)
    print(f"Topic '{TOPIC_NAME}' created successfully!")
    admin_client.close()
    
except TopicAlreadyExistsError:
    print(f"Topic '{TOPIC_NAME}' already exists.")
except Exception as e:
    print(f"Error: {e}")

## Kafka Producer Configuration

The producer serialises Python dictionaries into UTF-8 JSON bytes before sending them to Kafka. Producer settings such as batching and acknowledgements are configured to support reliable local streaming simulation.

In [ ]:
# Create Kafka producer with optimized settings
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    
    # JSON serialization: Convert Python dict → JSON string → bytes
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    
    # Compress messages using gzip (reduces network bandwidth)
    compression_type='gzip',
    
    # Wait for all replicas to acknowledge (strongest durability)
    acks='all',
    
    # Retry failed sends up to 3 times
    retries=3,
    
    # Client identifier for debugging/monitoring
    client_id='weather_producer'
)

print("Kafka producer initialized!")

## Weather Data Loading

The notebook loads `weather.csv`, sorts records by timestamp, and prepares the data for sequential publishing. The timestamp ordering helps preserve realistic event flow for the downstream streaming pipeline.

In [ ]:
# Load weather CSV file
weather_data = pd.read_csv("weather.csv")

# Sort by timestamp to ensure chronological order
if 'timestamp' in weather_data.columns:
    weather_data = weather_data.sort_values('timestamp').reset_index(drop=True)
# display data summary
print(f"Loaded {len(weather_data)} records")
print(f"Columns: {list(weather_data.columns)}")
print("\nFirst 3 records:")
print(weather_data.head(3))

## Producer Function Implementation

The main producer function streams weather records in small batches. Each record is converted to a JSON event, enriched with `weather_ts`, and published to Kafka to simulate a live sensor stream.

### Edge Case Handling

The producer handles missing values, timestamp conversion, JSON serialisation, and graceful termination. These checks make the notebook more stable when running in local development environments.

In [ ]:
def produce_weather_data():
    """
    Stream weather data to Kafka following Project 2B requirements.
    
    Process:
    1. Every 5 seconds, read 120 records (5 days × 24 hours)
    2. Add 'weather_ts' timestamp to each record
    3. Distribute timestamps: Day 1=T, Day 2=T+1, Day 3=T+2, etc.
    4. Send all records to Kafka topic
    5. Wait 5 seconds, then repeat
    
    This simulates a real-time weather sensor feed for downstream 
    Spark Streaming consumption and ML prediction.
    """
    
    # Initialize batch tracking
    pointer = 0                          # Current position in CSV file
    total_records = len(weather_data)     # Total records to stream
    batch_number = 0                      # Counter for batch tracking
    
    # Display streaming session info
    print("\n" + "="*70)
    print("STARTING PRODUCER")
    print("="*70)
    print(f"Total records: {total_records}")
    print(f"Estimated batches: {total_records // RECORDS_PER_BATCH}")
    print("="*70 + "\n")
    
    try:
        # Main streaming loop - continues until all data is sent
        while pointer < total_records:
            batch_number += 1
            
            # Get current Unix timestamp (seconds since epoch)
            # This serves as the base timestamp for this batch
            base_timestamp = int(time.time())
            
            # Extract next 5 days (120 records) from CSV
            # Handles edge case: last batch may have fewer records
            end_pointer = min(pointer + RECORDS_PER_BATCH, total_records)
            batch = weather_data.iloc[pointer:end_pointer].copy()
            
            # Log batch information
            print(f"[Batch {batch_number}] Records {pointer}-{end_pointer-1}")
            print(f"  Timestamp: {base_timestamp}")
            
            messages_sent = 0
            
            # Process each of the 5 days in the batch
            for day in range(DAYS_PER_BATCH):
                # Calculate record indices for this day
                day_start = day * RECORDS_PER_DAY           # Start: 0, 24, 48, 72, 96
                day_end = min(day_start + RECORDS_PER_DAY, len(batch))
                
                # Stop if we've processed all records in this batch
                if day_start >= len(batch):
                    break
                
                # Extract 24 hourly records for this day
                day_records = batch.iloc[day_start:day_end]
                
                # Calculate timestamp for this day
                # Day 1: T, Day 2: T+1, Day 3: T+2, Day 4: T+3, Day 5: T+4
                day_timestamp = base_timestamp + day
                
                # Send each hourly record to Kafka
                for idx, row in day_records.iterrows():
                    # Convert pandas Series to dictionary
                    record = row.to_dict()
                    
                    # Add weather_ts field (required by Spark Streaming in Spark streaming notebook)
                    record['weather_ts'] = day_timestamp
                    
                    try:
                        # Send message to Kafka topic
                        # Message is serialized to JSON automatically
                        producer.send(TOPIC_NAME, value=record)
                        messages_sent += 1
                        
                    except Exception as e:
                        print(f"    Error sending record: {e}")
            
            # Force all buffered messages to be sent immediately
            # Ensures batch is fully delivered before moving to next batch
            producer.flush()
            
            # Log batch completion
            print(f"  Sent {messages_sent} records")
            print(f"  Progress: {end_pointer}/{total_records} "
                  f"({(end_pointer/total_records)*100:.1f}%)\n")
            
            # Move pointer forward for next batch
            pointer = end_pointer
            
            # Wait 5 seconds before next batch (unless streaming is complete)
            if pointer < total_records:
                print(f"  Waiting {BATCH_INTERVAL} seconds...\n")
                time.sleep(BATCH_INTERVAL)
        
        # All data streamed successfully
        print("="*70)
        print("COMPLETED!")
        print(f"  Total batches: {batch_number}")
        print("="*70)
        
    except KeyboardInterrupt:
        # Handle user stopping the producer (Ctrl+C or Stop button)
        print("\nStopped by user")
        
    except Exception as e:
        # Handle unexpected errors
        print(f"\nError: {e}")
        raise
        
    finally:
        # Always close producer to release resources
        producer.close()
        print("Producer closed")

print(" Function defined")

## Execution and Validation

After execution, the producer sends weather events to Kafka in batches. The downstream Spark streaming notebook can then verify message arrival, parse the stream, apply the trained model, and produce prediction outputs.

In [ ]:
# Start producing data
# This will run until complete or you press Stop button
produce_weather_data()